In [102]:
from neo4j import GraphDatabase
import dotenv
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [103]:

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))


driver = GraphDatabase.driver(URI, auth=AUTH)

DIAG_LABEL = "Diagnostico"   
DIAG_NAME_PROP = "terminoEN"     
PACIENTE_LABEL = "Paciente"      

In [104]:
def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

In [105]:
# 1) ¿Hay nodos Paciente?
q_count_pat = f"MATCH (p:{PACIENTE_LABEL}) RETURN count(p) AS cnt"
cnt_pat = run_query(q_count_pat)[0]["cnt"]
print(f"Pacientes encontrados: {cnt_pat}")

# 2) Cargar nodos Diagnostico (id -> nombre)
q_nodos = f"MATCH (d:{DIAG_LABEL}) RETURN elementId(d) AS id, d.{DIAG_NAME_PROP} AS nombre"
nodos_raw = run_query(q_nodos)
id2name = {row["id"]: row["nombre"] or f"diag_{row['id']}" for row in nodos_raw}
print(f"Diagnósticos cargados: {len(id2name)}")

Pacientes encontrados: 1567
Diagnósticos cargados: 1746


In [106]:
import networkx as nx

# 1. Inicializar el grafo vacio
G = nx.Graph()

# 2. Añadir nodos al grafo con su etiqueta/nombre
for nid, nombre in id2name.items():
    G.add_node(nid, label=nombre)

# 3. Construir las conexiones y pesos desde Neo4j
if cnt_pat > 0:
    q_rels = f"""
    MATCH (p:{PACIENTE_LABEL})-[r]->(d:{DIAG_LABEL})
    RETURN elementId(p) AS pid, collect(DISTINCT elementId(d)) AS diagnosticos
    """
    rels = run_query(q_rels)
    
    for row in rels:
        diags = row["diagnosticos"] or []
        for i in range(len(diags)):
            for j in range(i + 1, len(diags)):
                a, b = diags[i], diags[j]
                if G.has_edge(a, b):
                    G[a][b]["weight"] += 1
                else:
                    G.add_edge(a, b, weight=1)
    print("Grafo de co-ocurrencia construido a partir de pacientes.")
else:
    # Corrección: uso de elementId en lugar del id() deprecado
    q_diag_links = f"""
    MATCH (d1:{DIAG_LABEL})-[r]->(d2:{DIAG_LABEL})
    RETURN elementId(d1) AS a, elementId(d2) AS b, type(r) AS relType, count(*) AS cnt
    """
    links = run_query(q_diag_links)
    for row in links:
        a, b, cnt = row["a"], row["b"], row["cnt"]
        if G.has_edge(a, b):
            G[a][b]["weight"] += cnt
        else:
            G.add_edge(a, b, weight=cnt)
    print("Grafo construido a partir de relaciones directas diagnóstico→diagnóstico.")

print(f"➜ Grafo original cargado: {G.number_of_nodes()} nodos | {G.number_of_edges()} aristas")

# 4. Comprobar distribución de pesos reales
pesos = [d.get("weight", 1) for u, v, d in G.edges(data=True)]
if pesos:
    print(f"➜ Peso máximo hallado: {max(pesos)}")
    print(f"➜ Peso promedio: {sum(pesos)/len(pesos):.2f}")

# 5. Filtrar aristas por debajo del umbral deseado
MIN_PESO = 40  # Ajusta según el peso máximo que te imprima arriba (ej. entre 3 y 10)

aristas_a_eliminar = [
    (u, v) for u, v, d in G.edges(data=True) if d.get("weight", 0) < MIN_PESO
]
G.remove_edges_from(aristas_a_eliminar)

# 6. Eliminar nodos que hayan quedado completamente aislados tras el filtro
G.remove_nodes_from(list(nx.isolates(G)))

print(f"➜ Grafo filtrado (MIN_PESO >= {MIN_PESO}): {G.number_of_nodes()} nodos | {G.number_of_edges()} aristas")

Grafo de co-ocurrencia construido a partir de pacientes.
➜ Grafo original cargado: 1746 nodos | 40657 aristas
➜ Peso máximo hallado: 419
➜ Peso promedio: 1.80
➜ Grafo filtrado (MIN_PESO >= 40): 31 nodos | 75 aristas


In [107]:
# 4) Normalizar / preparar pesos para algoritmos que los interpretan como 'coste'
# NetworkX interpreta 'weight' como la longitud de la arista (coste) para shortest-path.
# Como nuestras aristas con mayor 'weight' significan mayor co-ocurrencia (fuerza), para betweenness
# es preferible invertirlas: coste = 1/weight. Si weight==0 (no debería), evitamos división por 0.
for u, v, data in G.edges(data=True):
    w = data.get("weight", 1)
    data["inv_weight"] = 1.0 / w if w and w > 0 else float("inf")

print(f"Nodos en G: {G.number_of_nodes()}  Aristas en G: {G.number_of_edges()}")

Nodos en G: 31  Aristas en G: 75


In [108]:
# 5) Cálculo de métricas
print("Calculando Degree centrality...")
deg = nx.degree_centrality(G)  # normalizada por defecto (0..1)

print("Calculando Betweenness centrality (usa inv_weight para costes)...")
# weighted betweenness: usamos 'inv_weight' como coste para shortest paths
bet = nx.betweenness_centrality(G, weight='inv_weight', normalized=True)

print("Calculando Eigenvector centrality...")


def safe_eigenvector_centrality(graph, weight='weight'):
    if graph.number_of_nodes() == 0:
        return {}
    if graph.number_of_edges() == 0 or graph.number_of_nodes() == 1:
        return {n: 0.0 for n in graph.nodes()}

    components = list(nx.connected_components(graph))
    if len(components) == 1:
        try:
            return nx.eigenvector_centrality_numpy(graph, weight=weight, max_iter=1000)
        except Exception as e:
            print("eigenvector_centrality_numpy falló:", e, "Intentando power method...")
            try:
                return nx.eigenvector_centrality(graph, max_iter=10000, tol=1e-08, weight=weight)
            except Exception as e2:
                print("También falló eigenvector:", e2)
                return {n: 0.0 for n in graph.nodes()}

    centralities = {}
    for component in components:
        subgraph = graph.subgraph(component).copy()
        if subgraph.number_of_nodes() <= 1:
            for node in component:
                centralities[node] = 0.0
            continue

        try:
            comp_centrality = nx.eigenvector_centrality_numpy(subgraph, weight=weight, max_iter=1000)
        except Exception as e:
            print(f"eigenvector_centrality_numpy falló para componente de tamaño {subgraph.number_of_nodes()}:", e)
            try:
                comp_centrality = nx.eigenvector_centrality(subgraph, max_iter=10000, tol=1e-08, weight=weight)
            except Exception as e2:
                print("También falló eigenvector para este componente:", e2)
                comp_centrality = {n: 0.0 for n in subgraph.nodes()}

        for node, value in comp_centrality.items():
            centralities[node] = value

    return centralities


eig = safe_eigenvector_centrality(G, weight='weight')


Calculando Degree centrality...
Calculando Betweenness centrality (usa inv_weight para costes)...
Calculando Eigenvector centrality...


In [109]:
# 6) Crear DataFrame con resultados
rows = []
for node in G.nodes():
    rows.append({
        "node_id": node,
        "diagnostico": G.nodes[node].get("label", id2name.get(node, f"diag_{node}")),
        "degree": deg.get(node, 0.0),
        "betweenness": bet.get(node, 0.0),
        "eigenvector": eig.get(node, 0.0)
    })

df_metrics = pd.DataFrame(rows)
# ordenar por cada métrica y guardarlo
df_metrics.sort_values("betweenness", ascending=False).head(20)

# Guardar resultados
df_metrics.to_csv("diagnosticos_centralidades.csv", index=False)
print("Resultados guardados en diagnosticos_centralidades.csv")

Resultados guardados en diagnosticos_centralidades.csv


In [110]:
# 7) (Opcional) Mostrar top-N por cada métrica
N = 15
print("\nTop por Betweenness:")
print(df_metrics.sort_values("betweenness", ascending=False).head(N)[["diagnostico","betweenness","degree","eigenvector"]])

print("\nTop por Degree:")
print(df_metrics.sort_values("degree", ascending=False).head(N)[["diagnostico","degree","betweenness","eigenvector"]])

print("\nTop por Eigenvector:")
print(df_metrics.sort_values("eigenvector", ascending=False).head(N)[["diagnostico","eigenvector","degree","betweenness"]])




Top por Betweenness:
                                          diagnostico  betweenness    degree  \
13                       Otros tipos de esquizofrenia     0.880460  0.866667   
11                            Esquizofrenia paranoide     0.149425  0.433333   
10                                      Esquizofrenia     0.126437  0.400000   
14                     Esquizofrenia, no especificada     0.066667  0.366667   
30  Incumplimiento del paciente con otro tratamien...     0.057471  0.266667   
19                   Hipertensión esencial (primaria)     0.002299  0.266667   
0                     Hipotiroidismo, no especificado     0.000000  0.033333   
6                Abuso de alcohol, sin complicaciones     0.000000  0.066667   
5                     Hiperlipidemia, no especificada     0.000000  0.066667   
4                       Otros tipos de hiperlipidemia     0.000000  0.033333   
3                            Hipercolesterolemia pura     0.000000  0.033333   
9   Dependencia de

In [111]:
def get_edges():
    query = f"""
    MATCH (d1:Diagnostico)<-[:DIAGNOSTICO_ASOCIADO|DIAGNOSTICO_PSIQUIATRICO]-(:Paciente)-[:DIAGNOSTICO_ASOCIADO|DIAGNOSTICO_PSIQUIATRICO]->(d2:Diagnostico) 
    WHERE elementId(d1) < elementId(d2) 
    RETURN 
        coalesce(d1.terminoEN, elementId(d1)) AS source, 
        coalesce(d2.terminoEN, elementId(d2)) AS target,
        count(*) AS weight
    """
    with driver.session() as session:
        result = session.run(query)
        return pd.DataFrame(result.data())

edges_df = get_edges()
edges_df.head()
edges_df.to_csv("diagnosticos_edges.csv", index=False)

In [112]:
from pyvis.network import Network
from IPython.display import IFrame, HTML
import os

# Usar el grafo G y las métricas ya calculadas: bet, deg, eig
# 1. Inicializar red
net = Network(height="800px", width="100%", bgcolor="#ffffff", font_color="#111111")

# 2. Configurar nodos y aristas desde el grafo filtrado G
for node in G.nodes():
    label = G.nodes[node].get("label", str(node))

    # Recortar nombres muy largos para la vista principal (se verán completos en el hover)
    short_label = label[:25] + "..." if len(label) > 25 else label

    net.add_node(
        node,
        label=short_label,
        title=f"<b>{label}</b><br>Degree: {deg.get(node,0):.3f}<br>Betweenness: {bet.get(node,0):.3f}",
        value=deg.get(node, 0) * 500 + 5,  # Escalar tamaño según grado
        color="#e74c3c" if bet.get(node, 0) > 0.05 else "#3498db",
    )

for source, target, data in G.edges(data=True):
    weight = data.get("weight", 1)
    net.add_edge(source, target, value=weight, color="#cccccc")

# 3. Aplicar configuración de física y tipografía para evitar superposición
net.set_options("""
var options = {
  "nodes": {
    "font": {
      "size": 13,
      "face": "arial",
      "strokeWidth": 3,
      "strokeColor": "#ffffff"
    }
  },
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -80,
      "centralGravity": 0.01,
      "springLength": 120,
      "springConstant": 0.08,
      "avoidOverlap": 1
    },
    "maxVelocity": 50,
    "solver": "forceAtlas2Based",
    "timestep": 0.35,
    "stabilization": { "iterations": 200 }
  }
}
""")

# 4. Habilitar controles de física interactivos en la parte inferior del HTML

# 5. Guardar y mostrar
html_path = "grafo_diagnosticos.html"
net.write_html(html_path, open_browser=False)
display(IFrame(html_path, width="100%", height="800px"))
driver.close()
